In [2]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [3]:
from pathlib import Path
import json
import re
import sqlite3
from urllib.parse import unquote

import pandas as pd

In [4]:
import synth_extract.utils.sql_helpers as sql

#### Springer Nature 429 changes

In [5]:
db_path = "../data/central_papers.db"

with sqlite3.connect(db_path) as conn:
    springer_df = pd.read_sql_query(
        """
        SELECT *
        FROM papers
        WHERE canonical_source = ?
        """,
        conn,
        params=("springer_nature",),
    )

In [6]:
# Status counts, including unexpected or missing values
status_counts = (
    springer_df["download_status"]
    .fillna("missing")
    .value_counts(dropna=False)
)

total = len(springer_df)
successful = int(status_counts.get("success", 0))
failed = int(status_counts.get("failed", 0))
pending = int(status_counts.get("pending", 0))

stats_df = pd.DataFrame(
    {
        "status": ["total", "success", "failed", "pending"],
        "count": [total, successful, failed, pending],
    }
)

print(f"Total Springer Nature rows: {total:,}")
print(f"Successful:                {successful:,}")
print(f"Failed:                    {failed:,}")
print(f"Pending:                   {pending:,}")

Total Springer Nature rows: 14,913
Successful:                6,103
Failed:                    8,810
Pending:                   0


In [7]:
# source count distribution
springer_df["source_count"].value_counts()

source_count
1    13213
2      865
3      828
4        7
Name: count, dtype: int64

In [37]:
springer_df.columns

Index(['paper_id', 'paper_uid', 'identifier_type', 'identifier_value', 'doi',
       'arxiv_id', 'title', 'abstract', 'sources', 'source_count',
       'source_order', 'canonical_source', 'canonical_source_position',
       'download_status', 'attempt_count', 'attempted_sources',
       'failure_history', 'last_attempted_source', 'last_attempted_at',
       'last_error', 'downloaded_from', 'fulltext_path', 'fulltext_format',
       'downloaded_at', 'review_note', 'created_at', 'updated_at'],
      dtype='object')

In [15]:
success_df = springer_df[springer_df["download_status"] == "success"].copy()

print(f"Successful downloads: {len(success_df):,}")

# ------------------------------------------------------------------
# Full-text path / format completeness
# ------------------------------------------------------------------

missing_path = success_df["fulltext_path"].isna().sum()
missing_format = success_df["fulltext_format"].isna().sum()

print("\nFull-text information")
print("---------------------")
print(f"Missing fulltext_path:   {missing_path:,}")
print(f"Missing fulltext_format: {missing_format:,}")

# ------------------------------------------------------------------
# Attempt count distribution
# ------------------------------------------------------------------

print("\nAttempt count distribution")
print("--------------------------")
display(
    success_df["attempt_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("attempt_count")
    .reset_index(name="count")
)

# ------------------------------------------------------------------
# Download source distribution
# ------------------------------------------------------------------

print("\nDownloaded-from distribution")
print("----------------------------")
display(
    success_df["downloaded_from"]
    .value_counts(dropna=False)
    .rename_axis("downloaded_from")
    .reset_index(name="count")
)

# ------------------------------------------------------------------
# Unexpected last_error values
# ------------------------------------------------------------------

print("\nRows with last_error recorded")
print("----------------------------")
last_error_rows = success_df[success_df["last_error"].notna()]
print(f"{len(last_error_rows):,} rows")

if len(last_error_rows):
    display(
        last_error_rows[
            ["paper_uid", "doi", "attempt_count", "last_error"]
        ]
    )

# ------------------------------------------------------------------
# Unexpected failure_history values
# ------------------------------------------------------------------

def has_failure_history(value) -> bool:
    if pd.isna(value):
        return False

    if isinstance(value, list):
        return len(value) > 0

    if isinstance(value, str):
        value = value.strip()

        if value in {"", "[]", "null", "None"}:
            return False

        try:
            parsed = json.loads(value)
            return bool(parsed)
        except json.JSONDecodeError:
            return True

    return bool(value)

print("\nRows with faliure_history recorded")
print("----------------------------")

success_df["has_failure_history"] = (
    success_df["failure_history"].apply(has_failure_history)
)

failure_history_rows = success_df[
    success_df["has_failure_history"]
]

print(
    "Successful rows with non-empty failure history:",
    len(failure_history_rows),
)

if not failure_history_rows.empty:
    display(
        failure_history_rows[
            [
                "paper_uid",
                "doi",
                "attempt_count",
                "failure_history",
            ]
        ]
    )

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

print("\nSummary")
print("----------------------------")

summary = pd.DataFrame(
    {
        "metric": [
            "Successful rows",
            "Missing fulltext_path",
            "Missing fulltext_format",
            "Rows with last_error",
            "Rows with failure_history",
        ],
        "count": [
            len(success_df),
            missing_path,
            missing_format,
            len(last_error_rows),
            len(failure_history_rows),
        ],
    }
)

display(summary)

Successful downloads: 2,832

Full-text information
---------------------
Missing fulltext_path:   0
Missing fulltext_format: 0

Attempt count distribution
--------------------------


,attempt_count,count
0,1,2832



Downloaded-from distribution
----------------------------


,downloaded_from,count
0,springer_nature,2832



Rows with last_error recorded
----------------------------
0 rows

Rows with faliure_history recorded
----------------------------
Successful rows with non-empty failure history: 0

Summary
----------------------------


,metric,count
0,Successful rows,2832
1,Missing fulltext_path,0
2,Missing fulltext_format,0
3,Rows with last_error,0
4,Rows with failure_history,0


In [16]:
failed_df = springer_df[
    springer_df["download_status"] == "failed"
].copy()

print(f"Failed downloads: {len(failed_df):,}")


# ------------------------------------------------------------------
# Parse failure_history safely
# ------------------------------------------------------------------

def parse_failure_history(value):
    if value is None or pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        value = value.strip()

        if value in {"", "[]", "null", "None"}:
            return []

        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            return None

        return parsed if isinstance(parsed, list) else None

    return None


failed_df["parsed_failure_history"] = (
    failed_df["failure_history"].apply(parse_failure_history)
)

failed_df["failure_history_length"] = (
    failed_df["parsed_failure_history"].apply(
        lambda value: len(value) if isinstance(value, list) else pd.NA
    )
)


# ------------------------------------------------------------------
# Last-error distribution
# ------------------------------------------------------------------

print("\nLast-error distribution")
print("-----------------------")

last_error_distribution = (
    failed_df["last_error"]
    .fillna("<NULL>")
    .value_counts(dropna=False)
    .rename_axis("last_error")
    .reset_index(name="count")
)

display(last_error_distribution)


# ------------------------------------------------------------------
# Attempt-count distribution
# ------------------------------------------------------------------

print("\nAttempt-count distribution")
print("--------------------------")

attempt_distribution = (
    failed_df["attempt_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("attempt_count")
    .reset_index(name="count")
)

display(attempt_distribution)


# ------------------------------------------------------------------
# Failure-history length distribution
# ------------------------------------------------------------------

print("\nFailure-history length distribution")
print("-----------------------------------")

history_length_distribution = (
    failed_df["failure_history_length"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("failure_history_length")
    .reset_index(name="count")
)

display(history_length_distribution)


# ------------------------------------------------------------------
# Check that failure-history length matches attempt_count
# ------------------------------------------------------------------

failed_df["history_matches_attempt_count"] = (
    failed_df["failure_history_length"].astype("Int64")
    == failed_df["attempt_count"].astype("Int64")
)

print("\nFailure-history consistency")
print("---------------------------")

consistency_distribution = (
    failed_df["history_matches_attempt_count"]
    .value_counts(dropna=False)
    .rename_axis("history_matches_attempt_count")
    .reset_index(name="count")
)

display(consistency_distribution)


# ------------------------------------------------------------------
# Rows with multiple failed attempts
# ------------------------------------------------------------------

multiple_attempt_rows = failed_df[
    failed_df["attempt_count"].fillna(0) > 1
].copy()

print(
    "\nFailed rows with more than one attempt:",
    f"{len(multiple_attempt_rows):,}",
)

if not multiple_attempt_rows.empty:
    display(
        multiple_attempt_rows[
            [
                "paper_uid",
                "doi",
                "attempt_count",
                "failure_history_length",
                "last_error",
                "failure_history",
            ]
        ].head(20)
    )


# ------------------------------------------------------------------
# Rows where history length does not match attempt count
# ------------------------------------------------------------------

mismatch_rows = failed_df[
    failed_df["history_matches_attempt_count"] != True
].copy()

print(
    "\nRows where failure-history length does not match attempt_count:",
    f"{len(mismatch_rows):,}",
)

if not mismatch_rows.empty:
    display(
        mismatch_rows[
            [
                "paper_uid",
                "doi",
                "attempt_count",
                "failure_history_length",
                "last_error",
                "failure_history",
            ]
        ].head(20)
    )


# ------------------------------------------------------------------
# Malformed failure histories
# ------------------------------------------------------------------

malformed_history_rows = failed_df[
    failed_df["parsed_failure_history"].isna()
].copy()

print(
    "\nRows with malformed failure_history:",
    f"{len(malformed_history_rows):,}",
)

if not malformed_history_rows.empty:
    display(
        malformed_history_rows[
            [
                "paper_uid",
                "doi",
                "attempt_count",
                "failure_history",
            ]
        ].head(20)
    )


# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

summary = pd.DataFrame(
    {
        "metric": [
            "Failed rows",
            "Rows with NULL last_error",
            "Rows with empty failure_history",
            "Rows with multiple attempts",
            "History length matches attempt_count",
            "History length mismatch",
            "Malformed failure_history",
        ],
        "count": [
            len(failed_df),
            failed_df["last_error"].isna().sum(),
            (failed_df["failure_history_length"] == 0).sum(),
            len(multiple_attempt_rows),
            (failed_df["history_matches_attempt_count"] == True).sum(),
            len(mismatch_rows),
            len(malformed_history_rows),
        ],
    }
)

print("\nSummary")
print("-------")
display(summary)

Failed downloads: 6,348

Last-error distribution
-----------------------


,last_error,count
0,UNKNOWN_DOI | HTTP 404 | Fail | No data was fo...,2720
1,UNKNOWN_DOI | HTTP 200 | DOI '10.1007/s44174-0...,1
2,UNKNOWN_DOI | HTTP 200 | DOI '10.1007/s44174-0...,1
3,UNKNOWN_DOI | HTTP 200 | DOI '10.1007/s44174-0...,1
4,UNKNOWN_DOI | HTTP 200 | DOI '10.1007/s44174-0...,1
...,...,...
3624,UNKNOWN_DOI | HTTP 200 | DOI '10.1007/s11738-0...,1
3625,UNKNOWN_DOI | HTTP 200 | DOI '10.1007/s11738-0...,1
3626,UNKNOWN_DOI | HTTP 200 | DOI '10.1007/s11901-0...,1
3627,UNKNOWN_DOI | HTTP 200 | DOI '10.1007/s11901-0...,1



Attempt-count distribution
--------------------------


,attempt_count,count
0,1,6348



Failure-history length distribution
-----------------------------------


,failure_history_length,count
0,1,6348



Failure-history consistency
---------------------------


,history_matches_attempt_count,count
0,True,6348



Failed rows with more than one attempt: 0

Rows where failure-history length does not match attempt_count: 0

Rows with malformed failure_history: 0

Summary
-------


,metric,count
0,Failed rows,6348
1,Rows with NULL last_error,0
2,Rows with empty failure_history,0
3,Rows with multiple attempts,0
4,History length matches attempt_count,6348
5,History length mismatch,0
6,Malformed failure_history,0


In [42]:
# # 429 Rate Limited
# rate_limited = failed_df[
#     failed_df["last_error"].str.startswith("RATE_LIMITED", na=False)
# ]

# # 500 API Error
# api_error = failed_df[
#     failed_df["last_error"].str.startswith("API_ERROR", na=False)
# ]

# # 401 Authentication
# access_denied = failed_df[
#     failed_df["last_error"].str.startswith("ACCESS_DENIED", na=False)
# ]

# print(f"429: {len(rate_limited)}")
# print(f"500: {len(api_error)}")
# print(f"401: {len(access_denied)}")

# retry_uids = (
#     failed_df[
#         failed_df["last_error"].str.startswith(
#             ("RATE_LIMITED", "API_ERROR", "ACCESS_DENIED"),
#             na=False,
#         )
#     ]["paper_uid"]
#     .tolist()
# )

# print(f"Total papers to retry: {len(retry_uids)}")
# print(retry_uids)

In [43]:
# ## updating sql table

# if not retry_uids:
#     print("No papers need to be reset to pending.")
# else:
#     with sqlite3.connect(db_path) as conn:
#         cursor = conn.executemany(
#             """
#             UPDATE papers
#             SET download_status = 'pending',
#                 updated_at = CURRENT_TIMESTAMP
#             WHERE paper_uid = ?
#               AND canonical_source = 'springer_nature'
#               AND download_status = 'failed'
#             """,
#             [(paper_uid,) for paper_uid in retry_uids],
#         )

#         updated_count = cursor.rowcount

#     print(f"Rows reset to pending: {updated_count:,}")
#     print(f"Retry UIDs requested: {len(retry_uids):,}")


Updated contral table to restart springer nature downloads

#### Checking Elsevier and Wiley

In [17]:
db_path = "../data/central_papers.db"

with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query(
        """
        SELECT *
        FROM papers
        """,
        conn
    )

In [18]:
publishers = ["elsevier", "wiley"]

summary = []

for publisher in publishers:
    publisher_df = df[df["canonical_source"] == publisher]

    total = len(publisher_df)

    status_distribution = (
        publisher_df["download_status"]
        .fillna("missing")
        .value_counts()
        .rename_axis("download_status")
        .reset_index(name="count")
    )

    print(f"\n{'=' * 60}")
    print(f"{publisher.upper()}")
    print(f"{'=' * 60}")
    print(f"Total papers: {total:,}\n")

    display(status_distribution)

    # Save summary
    row = {"canonical_source": publisher, "total": total}
    for status in ["success", "failed", "pending"]:
        row[status] = int(
            status_distribution.loc[
                status_distribution["download_status"] == status,
                "count",
            ].sum()
        )
    summary.append(row)

summary_df = pd.DataFrame(summary)

print("\nSummary")
display(summary_df)


ELSEVIER
Total papers: 499,644



,download_status,count
0,pending,331500
1,success,161877
2,failed,6267



WILEY
Total papers: 189,418



,download_status,count
0,pending,172992
1,success,14171
2,failed,2255



Summary


,canonical_source,total,success,failed,pending
0,elsevier,499644,161877,6267,331500
1,wiley,189418,14171,2255,172992


In [19]:
publishers = ["elsevier", "wiley"]

for publisher in publishers:
    failed_df = df[
        (df["canonical_source"] == publisher)
        & (df["download_status"] == "failed")
    ]

    print(f"\n{'=' * 60}")
    print(f"{publisher.upper()} - Failed papers")
    print(f"{'=' * 60}")
    print(f"Total failed: {len(failed_df):,}\n")

    last_error_distribution = (
        failed_df["last_error"]
        .fillna("<NULL>")
        .value_counts(dropna=False)
        .rename_axis("last_error")
        .reset_index(name="count")
    )

    display(last_error_distribution)


ELSEVIER - Failed papers
Total failed: 6,267



,last_error,count
0,API_ERROR | HTTP 400 | INVALID_INPUT - View pa...,5462
1,UNKNOWN_DOI | HTTP 404 | RESOURCE_NOT_FOUND - ...,805



WILEY - Failed papers
Total failed: 2,255



,last_error,count
0,Unknown Doi | HTTP 404 | Wiley TDM returned HT...,1165
1,Invalid Doi | DOI failed format validation.,625
2,"Api Error | HTTP 500 | {""fault"":{""faultstring""...",250
3,"Api Error | HTTP 500 | {""fault"":{""faultstring""...",4
4,Api Error | HTTP 302 | Wiley TDM API error. Ch...,2
...,...,...
209,Access Denied | HTTP 403 | TDM access denied f...,1
210,Access Denied | HTTP 403 | TDM access denied f...,1
211,Access Denied | HTTP 403 | TDM access denied f...,1
212,Access Denied | HTTP 403 | TDM access denied f...,1


In [20]:
publishers = ["elsevier", "wiley"]

for publisher in publishers:
    failed_df = df[
        (df["canonical_source"] == publisher)
        & (df["download_status"] == "failed")
    ].copy()

    failed_df["error_type"] = (
        failed_df["last_error"]
        .fillna("<NULL>")
        .str.split("|")
        .str[0]
        .str.strip()
    )

    print(f"\n{'=' * 60}")
    print(f"{publisher.upper()} - Error type distribution")
    print(f"{'=' * 60}")

    display(
        failed_df["error_type"]
        .value_counts()
        .rename_axis("error_type")
        .reset_index(name="count")
    )


ELSEVIER - Error type distribution


,error_type,count
0,API_ERROR,5462
1,UNKNOWN_DOI,805



WILEY - Error type distribution


,error_type,count
0,Unknown Doi,1165
1,Invalid Doi,625
2,Api Error,259
3,Access Denied,201
4,Network Error,4
5,Storage Error,1


In [59]:
# wiley_unhandled = df[
#     (df["canonical_source"] == "wiley")
#     & (df["download_status"] == "failed")
#     & (
#         df["last_error"]
#         .fillna("")
#         .str.contains(
#             "Unhandled exception in downloader",
#             case=False,
#             regex=False,
#         )
#     )
# ]

# display(
#     wiley_unhandled[
#         [
#             "paper_uid",
#             "doi",
#             "attempt_count",
#             "last_error",
#             "failure_history",
#         ]
#     ]
# )

# uids = wiley_unhandled["paper_uid"].tolist()

# print(f"Found {len(uids):,} matching papers")
# print(uids)

In [ ]:
# # ID000007994 convert status to success

# with sqlite3.connect(db_path) as conn:
#     cursor = conn.execute(
#         """
#         UPDATE papers
#         SET download_status = 'success',
#             updated_at = CURRENT_TIMESTAMP
#         WHERE paper_uid = ?
#         """,
#         ("ID000007994",),
#     )

# print(f"Rows updated: {cursor.rowcount}")


In [21]:
uids = ["ID000016993", "ID000016994", "ID000363833", "ID000363834"]
df[df["paper_uid"].isin(uids)]["download_status"]

16992     success
16993     pending
363832    success
363833    pending
Name: download_status, dtype: object

looks good: very OK looking error distribution

#### PMC UPDATE

In [6]:
db_path = "../data/central_papers.db"

with sqlite3.connect(db_path) as conn:
    pmc_df = pd.read_sql_query(
        """
        SELECT *
        FROM papers
        WHERE canonical_source = ?
        """,
        conn,
        params=("europepmc",),
    )

In [7]:
pmc_df.head()

,paper_id,paper_uid,identifier_type,identifier_value,doi,arxiv_id,pmcid,title,abstract,sources,...,last_attempted_source,last_attempted_at,last_error,downloaded_from,fulltext_path,fulltext_format,downloaded_at,review_note,created_at,updated_at
0,1,ID000000001,pmcid,PMC3681841,10.1001/2013.jamaneurol.537,None,PMC3681841,C9orf72 hexanucleotide repeat expansions in cl...,<h4>Importance</h4>Hexanucleotide repeat expan...,"[""europepmc""]",...,None,None,None,None,None,None,None,None,2026-07-20 13:56:34,2026-07-23 16:44:36
1,4,ID000000004,pmcid,PMC2956589,10.1001/archderm.144.7.851,None,PMC2956589,Effect of increased pigmentation on the antifi...,"<h4>Objective</h4>To investigate the efficacy,...","[""europepmc""]",...,None,None,None,None,None,None,None,None,2026-07-20 13:56:34,2026-07-23 16:44:36
2,5,ID000000005,pmcid,PMC3787134,10.1001/archderm.144.7.902,None,PMC3787134,Acute skin eruptions that are positive for her...,<h4>Background</h4>Patients with stem cell tra...,"[""europepmc""]",...,None,None,None,None,None,None,None,None,2026-07-20 13:56:34,2026-07-23 16:44:36
3,6,ID000000006,pmcid,PMC3179848,10.1001/archdermatol.2010.86,None,PMC3179848,Pyoderma gangrenosum-like ulcer in a patient w...,<h4>Background</h4>Pyoderma gangrenosum-like u...,"[""europepmc""]",...,None,None,None,None,None,None,None,None,2026-07-20 13:56:34,2026-07-23 16:44:36
4,7,ID000000007,pmcid,PMC3346264,10.1001/archdermatol.2011.1413,None,PMC3346264,Viral-associated trichodysplasia: characteriza...,<h4>Background</h4>Viral-associated trichodysp...,"[""europepmc"",""s2orc""]",...,None,None,None,None,None,None,None,None,2026-07-20 13:56:34,2026-07-23 16:44:36


In [8]:
db_path = "../data/europepmc.db"

with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query(
        """
        SELECT *
        FROM papers_dup
        """,
        conn,
    )

In [7]:
df.head()

,pmcid,pmid,doi,title,journal,abstract
0,PMC13284822,42338903,10.1039/d6sc04630d,Alternating copolymerization of l-lactide and ...,Chemical science,The alternating copolymerization of chiral and...
1,PMC13254072,42271161,10.1038/s41467-026-73978-1,Synthesis of bundle copolymers as an emergent ...,Nature communications,Copolymers play a central role in soft materia...
2,PMC13296598,42263146,10.1021/acsnano.6c01299,A Robotic High-Throughput Grid-Search Platform...,ACS nano,We present a high-throughput experimental inve...
3,PMC13258949,42280619,10.3390/polym18111411,GMDH-Guided Variable Prioritization in PAGE Bl...,Polymers,The controlled synthesis of long hydrophobic b...
4,PMC13237634,42255128,10.1039/d6ra01918h,Copper-mediated stepwise polymerization of ben...,RSC advances,A novel synthetic approach for preparing well-...


In [9]:
pmc_df["identifier_type"].value_counts()

identifier_type
pmcid    255449
Name: count, dtype: int64

In [11]:
all(pmc_df["doi"].isin(df["doi"]))

True

this confirms all dois in central table are presnet in the europepmc table

In [83]:
## Now we need to update the indentivider type to pmcid in the central table and identifier value to the respective pmcid. pmcid can be fetched from europepmc.db (papers_dup table) 

In [ ]:
# # One-time migration: add PMCID support to the central papers table
# central_db_path = "../data/central_papers.db"

# with sqlite3.connect(central_db_path, timeout=30, isolation_level=None) as conn:
#     conn.execute("PRAGMA busy_timeout = 30000")
#     existing_columns = {
#         row[1] for row in conn.execute("PRAGMA table_info(papers)")
#     }

#     if "pmcid" in existing_columns:
#         print("Migration already applied: papers.pmcid exists.")
#     else:
#         conn.execute("BEGIN IMMEDIATE")
#         try:
#             old_count = conn.execute(
#                 "SELECT COUNT(*) FROM papers"
#             ).fetchone()[0]

#             conn.execute(
#                 """
#                 CREATE TABLE papers_new (
#                     paper_id INTEGER PRIMARY KEY,
#                     paper_uid TEXT NOT NULL UNIQUE,

#                     identifier_type TEXT NOT NULL
#                         CHECK (
#                             identifier_type IN ('doi', 'arxiv_id', 'pmcid')
#                         ),
#                     identifier_value TEXT NOT NULL UNIQUE,

#                     doi TEXT,
#                     arxiv_id TEXT,
#                     pmcid TEXT,

#                     title TEXT,
#                     abstract TEXT,
#                     sources TEXT NOT NULL,
#                     source_count INTEGER NOT NULL,
#                     source_order TEXT NOT NULL,
#                     canonical_source TEXT NOT NULL,
#                     canonical_source_position INTEGER NOT NULL DEFAULT 1,

#                     download_status TEXT NOT NULL DEFAULT 'pending'
#                         CHECK (
#                             download_status IN (
#                                 'pending', 'success', 'failed', 'exhausted'
#                             )
#                         ),
#                     attempt_count INTEGER NOT NULL DEFAULT 0,
#                     attempted_sources TEXT NOT NULL DEFAULT '[]',
#                     failure_history TEXT NOT NULL DEFAULT '[]',
#                     last_attempted_source TEXT,
#                     last_attempted_at TEXT,
#                     last_error TEXT,
#                     downloaded_from TEXT,
#                     fulltext_path TEXT,
#                     fulltext_format TEXT,
#                     downloaded_at TEXT,
#                     review_note TEXT,
#                     created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
#                     updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,

#                     CHECK (
#                         (identifier_type = 'doi' AND doi IS NOT NULL)
#                         OR
#                         (identifier_type = 'arxiv_id' AND arxiv_id IS NOT NULL)
#                         OR
#                         (identifier_type = 'pmcid' AND identifier_value IS NOT NULL)
#                     )
#                 )
#                 """
#             )

#             conn.execute(
#                 """
#                 INSERT INTO papers_new (
#                     paper_id, paper_uid, identifier_type, identifier_value,
#                     doi, arxiv_id, pmcid, title, abstract, sources,
#                     source_count, source_order, canonical_source,
#                     canonical_source_position, download_status, attempt_count,
#                     attempted_sources, failure_history, last_attempted_source,
#                     last_attempted_at, last_error, downloaded_from,
#                     fulltext_path, fulltext_format, downloaded_at, review_note,
#                     created_at, updated_at
#                 )
#                 SELECT
#                     paper_id, paper_uid, identifier_type, identifier_value,
#                     doi, arxiv_id, NULL, title, abstract, sources,
#                     source_count, source_order, canonical_source,
#                     canonical_source_position, download_status, attempt_count,
#                     attempted_sources, failure_history, last_attempted_source,
#                     last_attempted_at, last_error, downloaded_from,
#                     fulltext_path, fulltext_format, downloaded_at, review_note,
#                     created_at, updated_at
#                 FROM papers
#                 """
#             )

#             new_count = conn.execute(
#                 "SELECT COUNT(*) FROM papers_new"
#             ).fetchone()[0]
#             if new_count != old_count:
#                 raise RuntimeError(
#                     f"Row-count mismatch: old={old_count}, new={new_count}"
#                 )

#             conn.execute("DROP TABLE papers")
#             conn.execute("ALTER TABLE papers_new RENAME TO papers")
#             conn.execute(
#                 "CREATE UNIQUE INDEX idx_papers_uid ON papers(paper_uid)"
#             )
#             conn.execute(
#                 """
#                 CREATE UNIQUE INDEX idx_papers_identifier
#                 ON papers(identifier_type, identifier_value)
#                 """
#             )
#             conn.execute(
#                 """
#                 CREATE UNIQUE INDEX idx_papers_doi
#                 ON papers(doi) WHERE doi IS NOT NULL
#                 """
#             )
#             conn.execute(
#                 """
#                 CREATE UNIQUE INDEX idx_papers_arxiv_only
#                 ON papers(arxiv_id)
#                 WHERE identifier_type = 'arxiv_id' AND arxiv_id IS NOT NULL
#                 """
#             )
#             conn.execute(
#                 "CREATE INDEX idx_papers_download_status ON papers(download_status)"
#             )
#             conn.execute(
#                 "CREATE INDEX idx_papers_canonical_source ON papers(canonical_source)"
#             )
#             conn.execute("COMMIT")
#         except Exception:
#             conn.execute("ROLLBACK")
#             raise

#         integrity = conn.execute("PRAGMA integrity_check").fetchone()[0]
#         if integrity != "ok":
#             raise RuntimeError(f"Integrity check failed: {integrity}")

#         print(f"Migration complete: {new_count:,} rows preserved.")


Migration complete: 1,101,909 rows preserved.


In [ ]:
# # Populate PMCID identifiers from europepmc.db after running the migration
# central_db_path = "../data/central_papers.db"
# europepmc_db_path = "../data/europepmc.db"

# with sqlite3.connect(central_db_path, timeout=60) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")
#     conn.execute("ATTACH DATABASE ? AS europepmc", (europepmc_db_path,))
#     try:
#         # Copy the DOI-to-PMCID mapping into an indexed temporary table.
#         # DOI values are already normalized, so exact equality is sufficient.
#         conn.execute(
#             """
#             CREATE TEMP TABLE pmcid_map (
#                 doi TEXT PRIMARY KEY,
#                 mapped_pmcid TEXT NOT NULL UNIQUE
#             ) WITHOUT ROWID
#             """
#         )
#         conn.execute(
#             """
#             INSERT INTO pmcid_map (doi, mapped_pmcid)
#             SELECT doi, pmcid
#             FROM europepmc.papers_dup
#             WHERE doi IS NOT NULL
#               AND pmcid IS NOT NULL
#             """
#         )

#         unmatched_count = conn.execute(
#             """
#             SELECT COUNT(*)
#             FROM papers AS central
#             LEFT JOIN pmcid_map AS mapping ON mapping.doi = central.doi
#             WHERE central.canonical_source = 'europepmc'
#               AND mapping.doi IS NULL
#             """
#         ).fetchone()[0]
#         if unmatched_count:
#             raise RuntimeError(
#                 f"Europe PMC mapping is missing {unmatched_count:,} central DOI(s)"
#             )

#         changes_before = conn.total_changes
#         conn.execute(
#             """
#             UPDATE papers AS central
#             SET pmcid = mapping.mapped_pmcid,
#                 identifier_type = 'pmcid',
#                 identifier_value = mapping.mapped_pmcid,
#                 updated_at = CURRENT_TIMESTAMP
#             FROM pmcid_map AS mapping
#             WHERE central.canonical_source = 'europepmc'
#               AND central.doi = mapping.doi
#             """
#         )
#         updated_count = conn.total_changes - changes_before

#         remaining_count = conn.execute(
#             """
#             SELECT COUNT(*)
#             FROM papers
#             WHERE canonical_source = 'europepmc'
#               AND (identifier_type != 'pmcid' OR pmcid IS NULL)
#             """
#         ).fetchone()[0]
#         if remaining_count:
#             raise RuntimeError(
#                 f"PMCID update left {remaining_count:,} Europe PMC row(s) incomplete"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise
#     finally:
#         conn.execute("DROP TABLE IF EXISTS temp.pmcid_map")
#         conn.execute("DETACH DATABASE europepmc")

# print(f"Europe PMC rows updated to PMCID identifiers: {updated_count:,}")


Europe PMC rows updated to PMCID identifiers: 255,449
